# Lenormand B15.1 — Confirmed Latent Risk Test Ensemble

本 notebook 不训练新的27B。它部署已经由 Fold 1/2 确认的 `layer=63, C=0.01` latent risk readout：

1. 从B15现有hidden cache补建Fold 0 probe；
2. 三个冻结Task1 Qwen3.8-27B adapter分别对378条test帖子读取第64层；
3. 对三折row-normalized概率求平均；
4. 生成两个受控候选：
   - `SAFE_INDICATOR_ANCHOR`：旧Indicator保持不变，避免Evidence耦合风险；OOF Risk约`+0.0103`；
   - `FULL_LATENT_CUE_REPAIR`：使用完整latent预测，对新Indicator清空Evidence，对旧Indicator→非Indicator仅用高精度原文cue补Evidence；OOF Risk约`+0.0135`。

两个候选的Factors逐单元格保持不变。A100 80GB预计25–40分钟；每128个prompt保存到Drive，可断线恢复。


In [ ]:
#@title 0A. 新runtime安装依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.6.0,<1.8.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. Qwen3.8 kernels（之后重启session）
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation
print('Runtime → Restart session；重启后从第1格开始。')


In [ ]:
#@title 1. Drive、路径、当前最佳提交
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import gc, importlib, json, shutil, subprocess, sys, time

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
TEST_PATH = ROOT / 'leaderboard.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH = ROOT / 'ieee/train.xlsx'
if not TEST_PATH.exists(): TEST_PATH = ROOT / 'ieee/leaderboard.xlsx'
FOLD_SOURCE = ROOT / 'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'

B15_ROOT = ROOT / 'results/B15_LATENT_RISK_READOUT'
B151_ROOT = ROOT / 'results/B151_LATENT_STANDALONE_CONFIRMATION'
OUT = ROOT / 'results/B151_TEST_ENSEMBLE'
OUT.mkdir(parents=True, exist_ok=True)

# 当前线上0.7870/0.6542来源；若你另有更新后的最佳CSV，可直接修改这里。
SOURCE_SUBMISSION = ROOT / 'results/B7_TOP8_SPRINT/PROBES/03_EVIDENCE_PRECISION_045/Lenormand.csv'
if not SOURCE_SUBMISSION.exists():
    print('上传当前线上最佳 Lenormand.csv（0.7870 / 0.6542）')
    uploaded = files.upload()
    if 'Lenormand.csv' not in uploaded: raise FileNotFoundError('Lenormand.csv')
    SOURCE_SUBMISSION = OUT / 'SOURCE_Lenormand.csv'
    shutil.copy2('/content/Lenormand.csv', SOURCE_SUBMISSION)

MODULE_MARKERS = {
    'b1_experiments.py': None,
    'b4p_anchor_verifier.py': 'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
    'b4_task1_q38.py': 'TASK1_RUNTIME_REVISION = "2026-08-24.q38-full64-official-evidence-v4"',
    'b8_risk_only.py': 'B8R_RUNTIME_REVISION',
    'b7_top8_sprint.py': 'B7_RUNTIME_REVISION',
    'b15_latent_readout.py': 'def score_latent_test_fold(',
}
stale=[]
for name, marker in MODULE_MARKERS.items():
    path=ROOT/name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('上传并覆盖：', stale)
    uploaded=files.upload()
    for name in stale:
        if name not in uploaded: raise FileNotFoundError(name)
        shutil.copy2('/content/'+name, ROOT/name)

required=[TRAIN_PATH,TEST_PATH,FOLD_SOURCE,SOURCE_SUBMISSION,
          B15_ROOT/'fold_0/B15_FOLD_DECISION.json',
          B151_ROOT/'fold_1/B15_PROBE.joblib',B151_ROOT/'fold_2/B15_PROBE.joblib']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('缺少冻结产物：\n'+'\n'.join(missing))
sys.path.insert(0,str(ROOT))
print(subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],
    capture_output=True,text=True).stdout)
print({'source':str(SOURCE_SUBMISSION),'out':str(OUT),'free_gb':round(shutil.disk_usage(ROOT).free/2**30,2)})


In [ ]:
#@title 2. 环境、数据和源CSV审计
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

import b1_experiments as b1
import b4p_anchor_verifier as b4
import b4_task1_q38 as task1
import b8_risk_only as b8
import b7_top8_sprint as b7
import b15_latent_readout as b15
importlib.reload(b1);importlib.reload(b4);importlib.reload(task1);importlib.reload(b8);importlib.reload(b7);importlib.reload(b15)

gpu_gb=torch.cuda.get_device_properties(0).total_memory/2**30
kernel=b4.qwen35_kernel_status()
print({'transformers':transformers.__version__,'torch':torch.__version__,
       'sklearn':sklearn.__version__,'gpu_gb':gpu_gb,'kernel':kernel})
assert gpu_gb>=70
assert kernel['causal_conv1d'] and kernel['flash_linear_attention'], '运行0B后重启session'
torch.set_float32_matmul_precision('high')

bundle=b1.load_training_data(ROOT,TRAIN_PATH)
fold_saved=np.load(FOLD_SOURCE,allow_pickle=True)
assert fold_saved['row_ids'].astype(str).tolist()==bundle.row_ids.astype(str).tolist()
folds=fold_saved['folds'].astype(int)
test_corpus=b4.load_test_data(ROOT,TEST_PATH)
test_frame=test_corpus.frame.copy().fillna('')
source=pd.read_csv(SOURCE_SUBMISSION,dtype=str,keep_default_na=False)
source['row_id']=source.row_id.astype(str)
source_audit=b7.audit_submission(source,test_frame)
print({'test_rows':len(test_corpus.texts),'fold_sizes':np.bincount(folds).tolist(),
       'source_audit':source_audit})


In [ ]:
#@title 3. 用既有Fold-0 hidden cache补建probe（约3–12分钟）
fold0_probe=B15_ROOT/'fold_0/B15_PROBE.joblib'
if not fold0_probe.exists():
    cfg0=b15.LatentReadoutConfig(
        fold=0,selected_layers=(15,31,47,63),c_grid=(0.001,0.01,0.1),
        primary_blend_alpha=0.25,diagnostic_blend_alphas=(0.50,),
        extraction_batch_size=2,extraction_chunk_size=128,
    )
    result0=b15.run_latent_readout_fold(
        bundle,folds,ROOT,B15_ROOT/'fold_0',cfg0,
    )
    print({'refit_fold0_probe':str(fold0_probe),'chosen_layer':result0['chosen_layer'],
           'chosen_c':result0['chosen_c']})
assert fold0_probe.exists()

PROBES={
    0:fold0_probe,
    1:B151_ROOT/'fold_1/B15_PROBE.joblib',
    2:B151_ROOT/'fold_2/B15_PROBE.joblib',
}
print({fold:str(path) for fold,path in PROBES.items()})


In [ ]:
#@title 4. 三折test latent inference（断线后重跑本格）
fold_probabilities=[]
for fold in range(3):
    started=time.perf_counter()
    probability=b15.score_latent_test_fold(
        test_corpus=test_corpus,
        root=ROOT,
        output_dir=OUT/f'fold_{fold}',
        fold=fold,
        probe_path=PROBES[fold],
        extraction_batch_size=2,
        extraction_chunk_size=128,
    )
    fold_probabilities.append(probability)
    print({'fold':fold,'minutes':round((time.perf_counter()-started)/60,1),
           'risk_counts':np.bincount(np.argmax(probability,axis=1),minlength=4).tolist()})

ensemble_probability=np.mean(np.stack(fold_probabilities),axis=0)
latent_prediction=np.argmax(ensemble_probability,axis=1).astype(int)
np.savez_compressed(
    OUT/'B151_TEST_ENSEMBLE_PROBABILITIES.npz',
    row_ids=test_corpus.row_ids.astype(str),
    fold_probabilities=np.stack(fold_probabilities).astype(np.float32),
    ensemble_probability=ensemble_probability.astype(np.float32),
    prediction=latent_prediction.astype(np.int8),
)
print('ensemble counts:',dict(zip(b1.RISK_LABELS,np.bincount(latent_prediction,minlength=4).tolist())))


In [ ]:
#@title 5. 生成SAFE与FULL两个可上传候选
risk_to_id={name:i for i,name in enumerate(b1.RISK_LABELS)}
old_prediction=np.asarray([risk_to_id[value] for value in source.risk_level],dtype=int)
safe_prediction=np.where(old_prediction==0,0,latent_prediction).astype(int)

def build_candidate(name,prediction,repair_new_nonindicator):
    candidate=source.copy()
    candidate['risk_level']=[b1.RISK_LABELS[i] for i in prediction]
    evidence=candidate.evidence.astype(str).tolist()
    repair_rows=[]
    for row,(old,new) in enumerate(zip(old_prediction,prediction)):
        if int(new)==0:
            evidence[row]=''
        elif int(old)==0 and int(new)>0:
            phrase=b15.direct_cue_evidence(test_corpus.texts[row]) if repair_new_nonindicator else ''
            evidence[row]=phrase
            repair_rows.append({
                'row_id':str(test_corpus.row_ids[row]),'old_risk':b1.RISK_LABELS[int(old)],
                'new_risk':b1.RISK_LABELS[int(new)],'cue_evidence':phrase,
            })
    candidate['evidence']=evidence
    out_dir=OUT/name
    out_dir.mkdir(parents=True,exist_ok=True)
    path=out_dir/'Lenormand.csv'
    candidate.to_csv(path,index=False)
    reread=pd.read_csv(path,dtype=str,keep_default_na=False)
    audit=b7.audit_submission(reread,test_frame,source)
    audit.update({
        'candidate':name,'path':str(path),'sha256':b7.sha256(path),
        'factor_cells_identical':bool(np.array_equal(reread.factors.to_numpy(),source.factors.to_numpy())),
        'old_to_new_indicator':int(((old_prediction>0)&(prediction==0)).sum()),
        'old_indicator_to_new_risk':int(((old_prediction==0)&(prediction>0)).sum()),
        'cue_repairs_nonempty':int(sum(bool(row['cue_evidence']) for row in repair_rows)),
    })
    b7.json_dump(audit,out_dir/'AUDIT.json')
    pd.DataFrame(repair_rows).to_csv(out_dir/'INDICATOR_CROSSING_AUDIT.csv',index=False)
    return audit

audits=[
    build_candidate('01_B151_SAFE_INDICATOR_ANCHOR',safe_prediction,False),
    build_candidate('02_B151_FULL_LATENT_CUE_REPAIR',latent_prediction,True),
]
audit_frame=pd.DataFrame(audits)
display(audit_frame)
audit_frame.to_csv(OUT/'B151_TEST_CANDIDATES.csv',index=False)

assert all(row['factor_cells_identical'] for row in audits)
print('上传顺序：先SAFE；若Subtask1上升，再上传FULL。不要上传zip。')


In [ ]:
#@title 6. 打包报告并下载
package=Path('/content/B151_TEST_SUBMISSION_PACKAGE')
if package.exists():shutil.rmtree(package)
package.mkdir(parents=True)
for name in ('01_B151_SAFE_INDICATOR_ANCHOR','02_B151_FULL_LATENT_CUE_REPAIR'):
    shutil.copytree(OUT/name,package/name)
for name in ('B151_TEST_CANDIDATES.csv','B151_TEST_ENSEMBLE_PROBABILITIES.npz'):
    shutil.copy2(OUT/name,package/name)
archive=shutil.make_archive('/content/B151_TEST_SUBMISSION_PACKAGE','zip',package)
print('Package:',archive)
print('真正上传比赛的是候选目录内的 Lenormand.csv。')
files.download(archive)
